# Maximum likelihood: where loss functions come from

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The main question
Why does regression minimize the **squared** error and not the absolute value or the fourth power? Why does classification use **cross-entropy**? Both have the same answer: **maximum likelihood estimation**.

## Likelihood
Suppose a model has a parameter $\theta$. The **likelihood** says "if $\theta$ had this value, how probable would it be to see exactly this data?":

$$L(\theta)=P(\text{data}\mid\theta)=\prod_{i=1}^{n}P(x_i\mid\theta)$$

The best parameter is the one that makes the observed data **most probable**: $\hat\theta=\arg\max_\theta L(\theta)$.

## Example: a coin
We flipped a coin 10 times and got 7 heads. What is $\theta$ (the probability of heads)?

$$L(\theta)=\theta^7(1-\theta)^3$$

| $\theta$ | $L(\theta)$ | $\ln L(\theta)$ |
|---|---|---|
| 0.3 | 0.000075 | −9.50 |
| 0.5 | 0.000977 | −6.93 |
| 0.7 | 0.002224 | −6.11 |
| 0.9 | 0.000478 | −7.65 |

> Open the interactive lab for this part on the website: https://behrooztaheri.github.io/learn-python-ai/#/l/math-mle

## Solving with a derivative
Take the logarithm (products become sums and the location of the maximum does not move):

$$\ln L=7\ln\theta+3\ln(1-\theta)\ \Rightarrow\ \frac{d\ln L}{d\theta}=\frac7\theta-\frac{3}{1-\theta}=0\ \Rightarrow\ \hat\theta=\frac{7}{10}$$

In general $\hat\theta=k/n$: the simple ratio that intuition suggests.

## Example: the normal distribution
> [Left: three different bells for six data points. Right: the log-likelihood peaks at the sample mean.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_mle.png)

*Left: three different bells for six data points. Right: the log-likelihood peaks at the sample mean.*

If the data comes from $N(\mu,\sigma^2)$:

$$\ln L=-\frac{n}{2}\ln(2\pi\sigma^2)-\frac{1}{2\sigma^2}\sum_i(x_i-\mu)^2$$

| Parameter | MLE | For the data 4.2, 5.1, 5.6, 6.0, 6.3, 7.4 |
|---|---|---|
| $\hat\mu$ | $\frac1n\sum x_i$ | 5.767 |
| $\hat\sigma^2$ | $\frac1n\sum(x_i-\hat\mu)^2$ | 0.989 |

## From MLE to MSE
The regression assumption: $y_i=f_\theta(x_i)+\varepsilon_i$ with noise $\varepsilon_i\sim N(0,\sigma^2)$. So $y_i\sim N(f_\theta(x_i),\sigma^2)$ and:

$$\ln L(\theta)=\text{const}-\frac{1}{2\sigma^2}\sum_i\big(y_i-f_\theta(x_i)\big)^2$$

Maximizing this is exactly **minimizing the sum of squared errors**. So MSE is not an arbitrary choice: it follows directly from assuming normal noise.

## From MLE to cross-entropy
In binary classification the model outputs a probability $\hat{p}_i$ for class 1, and the label $y_i\in\{0,1\}$ is Bernoulli:

$$L=\prod_i\hat{p}_i^{\,y_i}(1-\hat{p}_i)^{1-y_i}\ \Rightarrow\ -\ln L=-\sum_i\big[y_i\ln\hat{p}_i+(1-y_i)\ln(1-\hat{p}_i)\big]$$

This is the **binary cross-entropy** (log loss).

| Assumption about the data | Resulting loss |
|---|---|
| Normal noise | MSE (squared error) |
| Laplace noise | MAE (absolute error) |
| Bernoulli labels | Binary cross-entropy |
| Categorical labels (K classes) | Multi-class cross-entropy |
| Poisson counts | Poisson loss |

## MAP: likelihood plus a prior belief
If we have a prior belief about the parameters ("the weights should be small"), we use Bayes:

$$\hat\theta_{\text{MAP}}=\arg\max_\theta\big[\ln P(\text{data}\mid\theta)+\ln P(\theta)\big]$$

| Prior on the weights | Extra term in the loss | Name |
|---|---|---|
| Normal $N(0,\tau^2)$ | $\lambda\sum w_j^2$ | Ridge (L2) |
| Laplace | $\lambda\sum\lvert w_j\rvert$ | Lasso (L1) |
| Uniform (no belief) | None | Plain MLE |

So regularization has a probabilistic origin too.



In [ ]:
import numpy as np
from scipy import stats, optimize

# 1) Coin: 7 heads in 10 flips
for theta in [0.3, 0.5, 0.7, 0.9]:
    L = theta ** 7 * (1 - theta) ** 3
    print(f"theta = {theta}:  L = {L:.6f}   ln L = {np.log(L):.2f}")
grid = np.linspace(0.01, 0.99, 981)
print("theta that maximizes ln L on a grid:", round(grid[np.argmax(7 * np.log(grid) + 3 * np.log(1 - grid))], 3))

# 2) Normal distribution: MLE is the sample mean and the 1/n variance
d = np.array([4.2, 5.1, 5.6, 6.0, 6.3, 7.4])
print("mu_hat =", round(d.mean(), 3), "  sigma2_hat =", round(d.var(), 3))
neg_ll = lambda p: -np.sum(stats.norm.logpdf(d, p[0], np.exp(p[1])))     # p[1] = ln(sigma)
best = optimize.minimize(neg_ll, [0.0, 0.0])
print("numerical optimizer:  mu =", round(best.x[0], 3), "  sigma2 =", round(np.exp(best.x[1]) ** 2, 3))

# 3) Regression: maximizing the Gaussian likelihood = least squares
rng = np.random.default_rng(1)
x = rng.uniform(0, 10, 50); y = 2 + 0.8 * x + rng.normal(0, 1, 50)
neg_ll_line = lambda p: -np.sum(stats.norm.logpdf(y, p[0] + p[1] * x, 1.0))
mle = optimize.minimize(neg_ll_line, [0.0, 0.0]).x
slope, intercept = np.polyfit(x, y, 1)
print("MLE (intercept, slope)          :", mle.round(4))
print("least squares (intercept, slope):", np.array([intercept, slope]).round(4))

# 4) Binary cross-entropy is the negative log-likelihood of Bernoulli labels
labels = np.array([1, 0, 1, 1])
probs  = np.array([0.9, 0.2, 0.6, 0.3])
likelihood = np.prod(np.where(labels == 1, probs, 1 - probs))
bce = -np.mean(labels * np.log(probs) + (1 - labels) * np.log(1 - probs))
print("likelihood =", round(likelihood, 4), "  -ln(likelihood)/n =", round(-np.log(likelihood) / 4, 4), "  BCE =", round(bce, 4))

### Exercise

Out of 20 visits to a page, 3 people bought. What is the maximum-likelihood estimate of the purchase rate? Compute $\ln L$ there.

In [ ]:
# Your answer here